In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
import matplotlib.pyplot as plt

import os

os.environ["NUMEXPR_MAX_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"

import numpy as np
from pathlib import Path

processed_path = Path("../data/splits/2026-08-16_processed_data_no_duplicates.npz")

data = np.load(processed_path)

print(data.files)

['X_train', 'X_val', 'X_test', 'X_id_train', 'X_id_val', 'X_id_test', 'X_err_train', 'X_err_val', 'X_err_test', 'y_train', 'y_val', 'y_test', 'y_id_train', 'y_id_val', 'y_id_test', 'y_ivar_train', 'y_ivar_val', 'y_ivar_test', 'gaia_wavelength', 'sdss_wavelength']


In [2]:
# Datos de Gaia
X_train = data["X_train"]
X_val = data["X_val"]
X_test = data["X_test"]

# Errores de Gaia
X_err_train = data["X_err_train"]
X_err_val = data["X_err_val"]
X_err_test = data["X_err_test"]

# IDs de Gaia
X_id_train = data["X_id_train"]
X_id_val = data["X_id_val"]
X_id_test = data["X_id_test"]

# Datos de SDSS
y_train = data["y_train"]
y_val = data["y_val"]
y_test = data["y_test"]

# IDs de SDSS
y_id_train = data["y_id_train"]
y_id_val = data["y_id_val"]
y_id_test = data["y_id_test"]

# Longitudes de onda
gaia_wavelength = data["gaia_wavelength"]
sdss_wavelength = data["sdss_wavelength"]

Aplicamos la normalización que mejores resultados dio en la red neuronal densa.

In [3]:
X_scale_train = np.nanmedian(np.abs(X_train), axis=1, keepdims=True)
X_scale_val = np.nanmedian(np.abs(X_val), axis=1, keepdims=True)
X_scale_test = np.nanmedian(np.abs(X_test), axis=1, keepdims=True)

y_scale_train = np.nanmedian(np.abs(y_train), axis=1, keepdims=True)
y_scale_val = np.nanmedian(np.abs(y_val), axis=1, keepdims=True)
y_scale_test = np.nanmedian(np.abs(y_test), axis=1, keepdims=True)

X_train_norm_median_spec = X_train / X_scale_train
X_val_norm_median_spec = X_val / X_scale_val
X_test_norm_median_spec = X_test / X_scale_test

y_train_norm_median_spec = y_train / y_scale_train
y_val_norm_median_spec = y_val / y_scale_val
y_test_norm_median_spec = y_test / y_scale_test

Preparamos los datos para la RNN añadiendo una tercera dimensión para las variables por cada paso (punto de la longitud de onda).

In [4]:
X_train_rnn = X_train_norm_median_spec[..., np.newaxis]
X_val_rnn = X_val_norm_median_spec[..., np.newaxis]
X_test_rnn = X_test_norm_median_spec[..., np.newaxis]

print("X_train_rnn:", X_train_rnn.shape)
print("X_val_rnn:", X_val_rnn.shape)
print("X_test_rnn:", X_test_rnn.shape)

X_train_rnn: (32656, 201, 1)
X_val_rnn: (4082, 201, 1)
X_test_rnn: (4082, 201, 1)


Definimos un primer modelo base para hacer pruebas.

In [5]:
n_gaia = X_train_rnn.shape[1]
n_features = X_train_rnn.shape[2]
n_sdss = y_train_norm_median_spec.shape[1]

model_rnn = models.Sequential([
    layers.Input(shape=(n_gaia, n_features)),

    layers.GRU(128, return_sequences=True),

    layers.GRU(128, return_sequences=False),

    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(1024),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(n_sdss)
])

model_rnn.compile(
    optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=["mae", "mse"]
)

model_rnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 201, 128)       │        50,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2666)           │     2,732,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,473,386 (13.25 MB)

 Trainable params: 3,473,386 (13.25 MB)

 Non-trainable params: 0 (0.00 B)

Definimos las mismas funciones que en el caso denso. 

In [6]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    min_delta=1e-4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

Ejecutamos el entrenamiento del modelo.

In [7]:
history_rnn = model_rnn.fit(
    X_train_rnn,
    y_train_norm_median_spec,
    validation_data=(X_val_rnn, y_val_norm_median_spec),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Epoch 1/50
511/511 ━━━━━━━━━━━━━━━━━━━━ 240s 460ms/step - loss: 0.1882 - mae: 0.4437 - mse: 0.4169 - val_loss: 0.0108 - val_mae: 0.0800 - val_mse: 0.0316 - learning_rate: 1.0000e-04
Epoch 2/50
511/511 ━━━━━━━━━━━━━━━━━━━━ 222s 434ms/step - loss: 0.0095 - mae: 0.0732 - mse: 0.0254 - val_loss: 0.0075 - val_mae: 0.0628 - val_mse: 0.0219 - learning_rate: 1.0000e-04
Epoch 3/50
511/511 ━━━━━━━━━━━━━━━━━━━━ 228s 445ms/step - loss: 0.0074 - mae: 0.0639 - mse: 0.0191 - val_loss: 0.0068 - val_mae: 0.0598 - val_mse: 0.0191 - learning_rate: 1.0000e-04
Epoch 4/50
511/511 ━━━━━━━━━━━━━━━━━━━━ 220s 431ms/step - loss: 0.0065 - mae: 0.0607 - mse: 0.0164 - val_loss: 0.0069 - val_mae: 0.0669 - val_mse: 0.0189 - learning_rate: 1.0000e-04
Epoch 5/50
511/511 ━━━━━━━━━━━━━━━━━━━━ 220s 431ms/step - loss: 0.0057 - mae: 0.0566 - mse: 0.0145 - val_loss: 0.0051 - val_mae: 0.0523 - val_mse: 0.0152 - learning_rate: 1.0000e-04
Epoch 6/50
511/511 ━━━━━━━━━━━━━━━━━━━━ 221s 433ms/step - loss: 0.0050 - mae: 0.0533 - mse

In [8]:
test_results_rnn = model_rnn.evaluate(
    X_test_rnn,
    y_test_norm_median_spec,
    verbose=1
)

128/128 ━━━━━━━━━━━━━━━━━━━━ 10s 80ms/step - loss: 0.0045 - mae: 0.0490 - mse: 0.0124


In [9]:
model_rnn_small = models.Sequential([
    layers.Input(shape=(n_gaia, n_features)),

    layers.GRU(128, return_sequences=False),

    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(1024),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(n_sdss)
])

model_rnn_small.compile(
    optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=["mae", "mse"]
)

model_rnn_small.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_2 (GRU)                     │ (None, 128)            │        50,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2666)           │     2,732,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,374,314 (12.87 MB)

 Trainable params: 3,374,314 (12.87 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history_rnn_small = model_rnn_small.fit(
    X_train_rnn,
    y_train_norm_median_spec,
    validation_data=(X_val_rnn, y_val_norm_median_spec),
    epochs=25,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Epoch 1/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 121s 233ms/step - loss: 0.2027 - mae: 0.4719 - mse: 0.4498 - val_loss: 0.0117 - val_mae: 0.0886 - val_mse: 0.0311 - learning_rate: 1.0000e-04
Epoch 2/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 93s 182ms/step - loss: 0.0102 - mae: 0.0788 - mse: 0.0249 - val_loss: 0.0085 - val_mae: 0.0682 - val_mse: 0.0235 - learning_rate: 1.0000e-04
Epoch 3/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 92s 180ms/step - loss: 0.0080 - mae: 0.0664 - mse: 0.0203 - val_loss: 0.0073 - val_mae: 0.0631 - val_mse: 0.0203 - learning_rate: 1.0000e-04
Epoch 4/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 96s 188ms/step - loss: 0.0073 - mae: 0.0643 - mse: 0.0178 - val_loss: 0.0069 - val_mae: 0.0613 - val_mse: 0.0190 - learning_rate: 1.0000e-04
Epoch 5/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 97s 190ms/step - loss: 0.0071 - mae: 0.0628 - mse: 0.0178 - val_loss: 0.0076 - val_mae: 0.0697 - val_mse: 0.0203 - learning_rate: 1.0000e-04
Epoch 6/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 93s 182ms/step - loss: 0.0066 - mae: 0.0611 - mse: 0.0

In [11]:
test_results_small_rnn = model_rnn_small.evaluate(
    X_test_rnn,
    y_test_norm_median_spec,
    verbose=1
)

128/128 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - loss: 0.0045 - mae: 0.0490 - mse: 0.0124


In [12]:
model_rnn_small_2 = models.Sequential([
    layers.Input(shape=(n_gaia, n_features)),

    layers.GRU(128, return_sequences=False),

    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(1024),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(n_sdss)
])

model_rnn_small_2.compile(
    optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=["mae", "mse"]
)

model_rnn_small_2.summary()

history_rnn_small_2 = model_rnn_small_2.fit(
    X_train_rnn,
    y_train_norm_median_spec,
    validation_data=(X_val_rnn, y_val_norm_median_spec),
    epochs=25,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_3 (GRU)                     │ (None, 128)            │        50,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_5 (LeakyReLU)       │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 2666)           │     2,732,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,374,314 (12.87 MB)

 Trainable params: 3,374,314 (12.87 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 92s 178ms/step - loss: 0.2029 - mae: 0.4736 - mse: 0.4488 - val_loss: 0.0116 - val_mae: 0.0830 - val_mse: 0.0316 - learning_rate: 1.0000e-04
Epoch 2/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 89s 173ms/step - loss: 0.0106 - mae: 0.0799 - mse: 0.0273 - val_loss: 0.0089 - val_mae: 0.0762 - val_mse: 0.0251 - learning_rate: 1.0000e-04
Epoch 3/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 86s 168ms/step - loss: 0.0080 - mae: 0.0672 - mse: 0.0201 - val_loss: 0.0083 - val_mae: 0.0742 - val_mse: 0.0226 - learning_rate: 1.0000e-04
Epoch 4/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 87s 170ms/step - loss: 0.0074 - mae: 0.0649 - mse: 0.0184 - val_loss: 0.0083 - val_mae: 0.0769 - val_mse: 0.0219 - learning_rate: 1.0000e-04
Epoch 5/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 99s 193ms/step - loss: 0.0070 - mae: 0.0628 - mse: 0.0172 - val_loss: 0.0074 - val_mae: 0.0657 - val_mse: 0.0199 - learning_rate: 1.0000e-04
Epoch 6/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 91s 177ms/step - loss: 0.0067 - mae: 0.0615 - mse: 0.01

In [13]:
model_rnn_small_3 = models.Sequential([
    layers.Input(shape=(n_gaia, n_features)),

    layers.GRU(128, return_sequences=False),
    layers.LayerNormalization(),

    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(1024),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(n_sdss)
])

model_rnn_small_3.compile(
    optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=["mae", "mse"]
)

model_rnn_small_3.summary()

history_rnn_small_3 = model_rnn_small_3.fit(
    X_train_rnn,
    y_train_norm_median_spec,
    validation_data=(X_val_rnn, y_val_norm_median_spec),
    epochs=25,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_4 (GRU)                     │ (None, 128)            │        50,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization             │ (None, 128)            │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_6 (LeakyReLU)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_7 (LeakyReLU)       │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 2666)           │     2,732,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,374,570 (12.87 MB)

 Trainable params: 3,374,570 (12.87 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 88s 169ms/step - loss: 0.1144 - mae: 0.3096 - mse: 0.2674 - val_loss: 0.0099 - val_mae: 0.0720 - val_mse: 0.0293 - learning_rate: 1.0000e-04
Epoch 2/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 88s 172ms/step - loss: 0.0091 - mae: 0.0693 - mse: 0.0240 - val_loss: 0.0077 - val_mae: 0.0662 - val_mse: 0.0214 - learning_rate: 1.0000e-04
Epoch 3/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 128s 251ms/step - loss: 0.0073 - mae: 0.0637 - mse: 0.0180 - val_loss: 0.0071 - val_mae: 0.0667 - val_mse: 0.0194 - learning_rate: 1.0000e-04
Epoch 4/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 113s 221ms/step - loss: 0.0067 - mae: 0.0616 - mse: 0.0162 - val_loss: 0.0066 - val_mae: 0.0603 - val_mse: 0.0182 - learning_rate: 1.0000e-04
Epoch 5/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 105s 206ms/step - loss: 0.0065 - mae: 0.0613 - mse: 0.0159 - val_loss: 0.0067 - val_mae: 0.0630 - val_mse: 0.0185 - learning_rate: 1.0000e-04
Epoch 6/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step - loss: 0.0064 - mae: 0.0606 - mse: 0.

In [14]:
model_rnn_small_4 = models.Sequential([
    layers.Input(shape=(n_gaia, n_features)),

    layers.GRU(64, return_sequences=True),
    layers.GRU(64, return_sequences=False),

    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(1024),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(n_sdss)
])

model_rnn_small_4.compile(
    optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=["mae", "mse"]
)

model_rnn_small_4.summary()

history_rnn_small_4 = model_rnn_small_4.fit(
    X_train_rnn,
    y_train_norm_median_spec,
    validation_data=(X_val_rnn, y_val_norm_median_spec),
    epochs=25,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_5 (GRU)                     │ (None, 201, 64)        │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_6 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 512)            │        33,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_8 (LeakyReLU)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_9 (LeakyReLU)       │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 2666)           │     2,732,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,329,066 (12.70 MB)

 Trainable params: 3,329,066 (12.70 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 81s 155ms/step - loss: 0.2115 - mae: 0.4877 - mse: 0.4692 - val_loss: 0.0121 - val_mae: 0.0880 - val_mse: 0.0339 - learning_rate: 1.0000e-04
Epoch 2/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 78s 152ms/step - loss: 0.0109 - mae: 0.0804 - mse: 0.0278 - val_loss: 0.0085 - val_mae: 0.0671 - val_mse: 0.0243 - learning_rate: 1.0000e-04
Epoch 3/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 79s 154ms/step - loss: 0.0081 - mae: 0.0664 - mse: 0.0203 - val_loss: 0.0078 - val_mae: 0.0641 - val_mse: 0.0217 - learning_rate: 1.0000e-04
Epoch 4/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 77s 150ms/step - loss: 0.0075 - mae: 0.0637 - mse: 0.0190 - val_loss: 0.0069 - val_mae: 0.0598 - val_mse: 0.0192 - learning_rate: 1.0000e-04
Epoch 5/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 75s 147ms/step - loss: 0.0068 - mae: 0.0609 - mse: 0.0165 - val_loss: 0.0066 - val_mae: 0.0595 - val_mse: 0.0183 - learning_rate: 1.0000e-04
Epoch 6/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 77s 150ms/step - loss: 0.0065 - mae: 0.0601 - mse: 0.01

In [15]:
model_rnn_5 = models.Sequential([
    layers.Input(shape=(n_gaia, n_features)),

    layers.GRU(128, return_sequences=True),
    layers.GRU(128, return_sequences=False),

    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(1024, activation="elu"),

    layers.Dense(1024),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(1024, activation="elu"),

    layers.Dense(n_sdss)
])

model_rnn_5.compile(
    optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=["mae", "mse"]
)

model_rnn_5.summary()

history_rnn_5 = model_rnn_5.fit(
    X_train_rnn,
    y_train_norm_median_spec,
    validation_data=(X_val_rnn, y_val_norm_median_spec),
    epochs=25,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_7 (GRU)                     │ (None, 201, 128)       │        50,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_8 (GRU)                     │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_10 (LeakyReLU)      │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 1024)           │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1024)           │     1,049,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_11 (LeakyReLU)      │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 1024)           │     1,049,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 2666)           │     2,732,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,572,586 (21.26 MB)

 Trainable params: 5,572,586 (21.26 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 189s 366ms/step - loss: 0.1382 - mae: 0.3510 - mse: 0.3046 - val_loss: 0.0153 - val_mae: 0.1061 - val_mse: 0.0403 - learning_rate: 1.0000e-04
Epoch 2/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 173s 338ms/step - loss: 0.0130 - mae: 0.0905 - mse: 0.0322 - val_loss: 0.0097 - val_mae: 0.0810 - val_mse: 0.0263 - learning_rate: 1.0000e-04
Epoch 3/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 178s 349ms/step - loss: 0.0087 - mae: 0.0727 - mse: 0.0214 - val_loss: 0.0077 - val_mae: 0.0683 - val_mse: 0.0210 - learning_rate: 1.0000e-04
Epoch 4/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 182s 356ms/step - loss: 0.0072 - mae: 0.0661 - mse: 0.0183 - val_loss: 0.0063 - val_mae: 0.0624 - val_mse: 0.0177 - learning_rate: 1.0000e-04
Epoch 5/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 179s 351ms/step - loss: 0.0063 - mae: 0.0621 - mse: 0.0149 - val_loss: 0.0057 - val_mae: 0.0582 - val_mse: 0.0164 - learning_rate: 1.0000e-04
Epoch 6/25
511/511 ━━━━━━━━━━━━━━━━━━━━ 183s 359ms/step - loss: 0.0057 - mae: 0.0598 - mse